   # 1.0 Data Cleaning and Staging
   In this notebook we extract the raw Kaggle transactional data for "Online Retail II UCI", remove PII (if there is any), and stage the data for feature engineering. Information on the data set is available on the following path:
   https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci/data

   It is an OLTP type information set for 2 years worth of orders.

   
   First, we test the virtual environment is set up correctly:

In [1]:
   import pandas as pd
   print("Pandas is working")

Pandas is working


# Extracting the information from Kaggle

To download data via Kaggle's API we need to install kagglehub. For reference I already installed {pandas, numpy, matplotlib, seaborn, scikit-learn, ipykernel, ipywidgets} in the virtual (.venv) environment. Required packages are listed in requirements.txt

We proceed to download the file (to cache):

In [5]:
import kagglehub
import shutil
import os

print("Downloading dataset from Kaggle...")
cache_path = kagglehub.dataset_download("mashlyn/online-retail-ii-uci")
print(f"Downloaded to cache: {cache_path}")

100%|██████████| 14.5M/14.5M [00:01<00:00, 8.85MB/s]

Extracting files...


Downloaded to cache: /Users/sachaplaye/.cache/kagglehub/datasets/mashlyn/online-retail-ii-uci/versions/3


#### We locate the .csv file in the cache folder

In [6]:
files = os.listdir(cache_path)
csv_filename =[f for f in files if f.endswith('.csv')][0]
source_file = os.path.join(cache_path, csv_filename)

#### Set the path for the destination folder IE data/raw

In [7]:
destination_file = "../data/raw/online_retail_II.csv"

#### Transfer the source file to raw and check a sample of 10 lines

In [8]:
shutil.copy(source_file, destination_file)
print(f"Success! Raw data saved to: {destination_file}")

df_raw = pd.read_csv(destination_file)
df_raw.sample(10)

Success! Raw data saved to: ../data/raw/online_retail_II.csv


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
85083,497272,22196,SMALL HEART MEASURING SPOONS,24,2010-02-08 13:50:00,0.85,17160.0,United Kingdom
989506,576076,84596B,SMALL DOLLY MIX DESIGN ORANGE BOWL,5,2011-11-13 16:18:00,0.42,14382.0,United Kingdom
935107,572066,22219,LOVEBIRD HANGING DECORATION WHITE,12,2011-10-20 13:07:00,0.85,15159.0,United Kingdom
237772,512425,21034,REX CASH+CARRY JUMBO SHOPPER,2,2010-06-15 14:42:00,0.95,14723.0,United Kingdom
404427,528104,22797,CHEST OF DRAWERS GINGHAM HEART,1,2010-10-20 13:42:00,16.95,13842.0,United Kingdom
91129,497954,21482,MUSHROOM BLUE HOT WATER BOTTLE,2,2010-02-15 12:50:00,2.95,16225.0,United Kingdom
238882,512523,22424,ENAMEL BREAD BIN CREAM,2,2010-06-16 11:49:00,12.75,13769.0,United Kingdom
477762,534257,10135,COLOURING PENCILS BROWN TUBE,4,2010-11-22 11:09:00,1.25,14419.0,United Kingdom
339365,522464,85116,BLACK CANDELABRA T-LIGHT HOLDER,8,2010-09-14 17:50:00,1.66,NaN,United Kingdom
916213,570597,23403,LETTER HOLDER HOME SWEET HOME,4,2011-10-11 11:49:00,3.75,16837.0,United Kingdom


# Cleaning the data prior to staging

- So on the left I see the pandas index, stockcodes that are varchar (it looks like maybe they should be int). 
- Customer ID is to 1dp and also it needs to be hashed for GDPR, there is no other PII, although we can check across all the rows.
- InvoiceDate will need to be datetime format for rfm.
- It will be good to check for typo's in the description field and if StockCode - Description is one to one.
- I see a NaN in Customer ID, so it would be good to see if that is an issue, if there are NaN's elsewhere and how to handle them?
- How does Invoice field relate to Quantity and is it unique?
- There appears to be a trove of valuable information in the Description field that can be recategorised into useful grouping tags for feature engineering.

#### The next two steps prior to any cleansing, is to check columns, data types and missing values:

In [10]:
df_raw.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='str')

In [11]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB
